# Vol Surface PCA Engine — Demo Notebook

End-to-end walkthrough: data loading → preprocessing → PCA (Static / Rolling / Iterated) → P&L attribution → model comparison.

**Replace** the `LiveDataLoader` stub with your actual Bloomberg / internal API calls.

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from vol_pca_engine import (
    VolDataLoader,
    GridSpec, SurfaceConfig, SurfacePreprocessor,
    compute_covariance,
    StaticPCA, RollingPCA, IteratedPCA,
    PnLAttributor, PCAModelComparison,
)

plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'grid.alpha': 0.5, 'lines.linewidth': 1.8,
    'font.size': 11})

C_BLUE   = '#58a6ff'
C_ORANGE = '#f0883e'
C_GREEN  = '#3fb950'
C_RED    = '#f85149'
C_GRAY   = '#8b949e'

## 2 · Define grids and configuration

Edit these lists to match your actual surface pillars.

In [ ]:
# ── Swaption grid ────────────────────────────────────────────────────────
SWP_EXPIRIES = ['2D','1M','3M','6M','9M','1Y','18M','2Y','3Y','4Y','5Y','7Y','10Y','15Y','20Y']
SWP_TENORS   = ['1Y','2Y','3Y','4Y','5Y','7Y','10Y','15Y','20Y','30Y']

# ── Cap floor grid ───────────────────────────────────────────────────────
CAP_EXPIRIES = ['2D','1M','2M','3M','6M','9M','1Y','18M','2Y','3Y',
                '4Y','5Y','6Y','7Y','8Y','9Y','10Y','12Y','15Y','20Y','25Y','30Y']
CAP_TENORS   = ['1M','3M','6M','12M']

# ── Model config ─────────────────────────────────────────────────────────
cfg = SurfaceConfig(
    swaption_grid       = GridSpec(expiries=SWP_EXPIRIES, tenors=SWP_TENORS),
    capfloor_grid       = GridSpec(expiries=CAP_EXPIRIES, tenors=CAP_TENORS),
    cap_injection_tenor = '1Y',   # swaption pillar that absorbs cap vega
    separate_pca        = True,   # recommended: run cap/swaption PCA separately
    unit_divisor        = 1000.0, # convert bps → kbps
)

prep = SurfacePreprocessor(cfg)
print('Grid: swaptions', len(SWP_EXPIRIES), 'x', len(SWP_TENORS),
      '=', len(SWP_EXPIRIES)*len(SWP_TENORS), 'features')

## 3 · Data loader

Implement `LiveDataLoader` below for your system.
The `MockDataLoader` generates synthetic correlated vol changes for testing.

In [ ]:
class LiveDataLoader(VolDataLoader):
    """
    TODO: implement these three methods using your data source.

    Example (Bloomberg via blpapi):
        def get_vol_changes(self, start_date, end_date, surface):
            # fields  = ['IVOL_MID'] per tenor/expiry
            # service = blp.BlpDataService()
            # raw     = service.bdh(tickers, 'IVOL_MID', start_date, end_date)
            # return  raw.diff().dropna()  # daily changes
            raise NotImplementedError
    """

    def get_vol_changes(self, start_date, end_date, surface):
        raise NotImplementedError('Connect your data source here')

    def get_vega(self, as_of, surface):
        raise NotImplementedError('Connect your data source here')

    def get_spot_vol_change(self, as_of, surface):
        raise NotImplementedError('Connect your data source here')


# ── Mock loader (synthetic data — replace with LiveDataLoader) ────────────
class MockDataLoader(VolDataLoader):
    """Generates correlated synthetic vol changes for testing."""

    def __init__(self, seed=42):
        self.seed = seed

    def _make_correlated_surface(self, T, expiries, tenors, scale=2.0):
        np.random.seed(self.seed)
        n  = len(expiries) * len(tenors)
        # Low-rank correlation: 3 latent factors (level, slope, curvature)
        F  = np.random.randn(T, 3)
        B  = np.random.randn(3, n) * scale
        X  = F @ B + np.random.randn(T, n) * 0.3
        dates = pd.bdate_range('2022-01-03', periods=T)
        cols  = pd.MultiIndex.from_product(
            [expiries, tenors], names=['expiry', 'tenor']
        )
        return pd.DataFrame(X, index=dates, columns=cols)

    def get_vol_changes(self, start_date, end_date, surface='swaptions'):
        expiries = SWP_EXPIRIES if surface == 'swaptions' else CAP_EXPIRIES
        tenors   = SWP_TENORS   if surface == 'swaptions' else CAP_TENORS
        T = len(pd.bdate_range(start_date, end_date))
        return self._make_correlated_surface(T, expiries, tenors)

    def get_vega(self, as_of, surface='swaptions'):
        np.random.seed(1)
        expiries = SWP_EXPIRIES if surface == 'swaptions' else CAP_EXPIRIES
        tenors   = SWP_TENORS   if surface == 'swaptions' else CAP_TENORS
        data = np.random.randn(len(expiries), len(tenors)) * 50
        return pd.DataFrame(data, index=expiries, columns=tenors)

    def get_spot_vol_change(self, as_of, surface='swaptions'):
        np.random.seed(int(pd.Timestamp(as_of).timestamp()) % 9999)
        expiries = SWP_EXPIRIES if surface == 'swaptions' else CAP_EXPIRIES
        tenors   = SWP_TENORS   if surface == 'swaptions' else CAP_TENORS
        data = np.random.randn(len(expiries), len(tenors)) * 3.0
        return pd.DataFrame(data, index=expiries, columns=tenors)


loader = MockDataLoader()   # ← swap to LiveDataLoader() when ready
print('Loader ready:', type(loader).__name__)

## 4 · Load historical data

Fetch the full history of daily vol changes and flatten to a `(T × n_features)` matrix.

In [ ]:
TRAIN_START = '2022-01-03'
TRAIN_END   = '2025-12-31'
TODAY       = '2026-04-09'   # the date you want to run attribution for

# ── Swaptions ─────────────────────────────────────────────────────────────
print('Loading swaption vol change history …')
swp_history_raw = loader.get_vol_changes(TRAIN_START, TRAIN_END, 'swaptions')
X_swp   = prep.flatten_history(swp_history_raw, 'swaptions')
dates   = swp_history_raw.index
T, n_f  = X_swp.shape
print(f'  {T} days × {n_f} features')

# ── Cap floors (optional — only needed if separate_pca=True) ─────────────
if cfg.separate_pca:
    print('Loading cap floor vol change history …')
    cap_history_raw = loader.get_vol_changes(TRAIN_START, TRAIN_END, 'capfloor')
    X_cap   = prep.flatten_history(cap_history_raw, 'capfloor')
    print(f'  {X_cap.shape[0]} days × {X_cap.shape[1]} cap features')

# ── Today's surfaces ──────────────────────────────────────────────────────
vega_swp_raw  = loader.get_vega(TODAY, 'swaptions')
dv_swp_raw    = loader.get_spot_vol_change(TODAY, 'swaptions')

vega_swp  = prep.reindex(vega_swp_raw,  'swaptions')   # aligned to model grid
dv_swp    = prep.reindex(dv_swp_raw,    'swaptions')

if cfg.separate_pca:
    vega_cap_raw  = loader.get_vega(TODAY, 'capfloor')
    dv_cap_raw    = loader.get_spot_vol_change(TODAY, 'capfloor')
    vega_cap  = prep.reindex(vega_cap_raw, 'capfloor')
    dv_cap    = prep.reindex(dv_cap_raw,   'capfloor')

print('\nData ready.')

## 5 · Eigenspectrum — how many PCs do we need?

Before fitting models, check variance explained to pick `n_components`.

In [ ]:
from vol_pca_engine import compute_covariance

def plot_eigenspectrum(X, title='Eigenspectrum', max_pcs=25,
                       cov_method='ledoit_wolf'):
    Xc   = X - X.mean(axis=0)
    cov  = compute_covariance(Xc, method=cov_method)
    evals, _ = np.linalg.eigh(cov)
    evals    = np.sort(evals)[::-1]
    evals    = np.maximum(evals, 0)
    total    = evals.sum()
    evr      = evals / total
    cumevr   = np.cumsum(evr)
    n        = min(max_pcs, len(evals))

    n95 = int(np.searchsorted(cumevr, 0.95)) + 1
    n99 = int(np.searchsorted(cumevr, 0.99)) + 1

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    # Scree
    ax = axes[0]
    colors = [C_ORANGE if i < 5 else C_BLUE for i in range(n)]
    ax.bar(range(1, n+1), evr[:n]*100, color=colors, alpha=0.85, width=0.7)
    ax.axvline(5.5,   color=C_ORANGE, ls='--', lw=1.2, label='Current 5 PCs')
    ax.axvline(n95+.5,color=C_GREEN,  ls=':',  lw=1.2, label=f'95% @ PC{n95}')
    ax.set_xlabel('PC'); ax.set_ylabel('% variance'); ax.set_title('Scree')
    ax.legend(fontsize=9); ax.grid(True, axis='y')

    # Cumulative
    ax = axes[1]
    ax.plot(range(1, n+1), cumevr[:n]*100, color=C_BLUE, marker='o', ms=3)
    ax.axhline(95, color=C_GREEN,  ls=':', lw=1.2, label='95%')
    ax.axhline(99, color=C_ORANGE, ls=':', lw=1.2, label='99%')
    ax.axvline(n95, color=C_GREEN,  ls=':', lw=1.0, alpha=0.7)
    ax.axvline(n99, color=C_ORANGE, ls=':', lw=1.0, alpha=0.7)
    for level, n_at in [(95, n95), (99, n99)]:
        ax.annotate(f'{level}% @ PC{n_at}', xy=(n_at, level),
                    xytext=(n_at+1, level-6), color=C_GREEN if level==95 else C_ORANGE,
                    fontsize=9, arrowprops=dict(arrowstyle='->', lw=0.8,
                    color=C_GREEN if level==95 else C_ORANGE))
    ax.set_xlabel('Number of PCs'); ax.set_ylabel('Cumulative %')
    ax.set_title('Cumulative variance'); ax.legend(fontsize=9); ax.grid(True)

    plt.tight_layout()
    plt.show()
    return n95, n99

n95_swp, n99_swp = plot_eigenspectrum(X_swp, 'Swaptions — eigenspectrum')
if cfg.separate_pca:
    n95_cap, n99_cap = plot_eigenspectrum(X_cap, 'Cap floors — eigenspectrum')
print(f'\nRecommended PCs: swaptions={n95_swp}, caps={n95_cap if cfg.separate_pca else "N/A"}')

## 6 · Static PCA

Fitted once on the full history. Try different covariance estimators and compare.

In [ ]:
# ── Fit with multiple covariance estimators ───────────────────────────────
static_models = {}
for cov_name in ['ledoit_wolf', 'ewma_lw', 'oas']:
    m = StaticPCA(
        n_components    = 5,        # minimum; auto_select_n will raise this
        cov_method      = cov_name,
        ewma_lambda     = 0.96,
        auto_select_n   = True,
        target_variance = 0.95,
    )
    m.fit(X_swp)
    static_models[cov_name] = m
    res = m.result_
    print(f'{cov_name:15s}  →  {res.n_components} PCs  |  '
          f'{res.cumulative_variance_ratio[-1]*100:.2f}% var')

# ── Use the best estimator going forward (ledoit_wolf recommended) ────────
BEST_COV   = 'ledoit_wolf'
static_pca = static_models[BEST_COV]
res_static = static_pca.result_

In [ ]:
def plot_loadings(result, title, n_show=5):
    n  = min(n_show, result.n_components)
    L  = result.loadings[:n]
    nE = len(SWP_EXPIRIES)
    nT = len(SWP_TENORS)

    fig, axes = plt.subplots(1, n, figsize=(3.5*n, 3.5))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    if n == 1: axes = [axes]

    for k, ax in enumerate(axes):
        mat  = L[k].reshape(nE, nT)
        vmax = np.abs(mat).max()
        im   = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                         aspect='auto')
        ax.set_title(f'PC{k+1}  ({result.explained_variance_ratio[k]*100:.1f}%)',
                     fontsize=10)
        ax.set_xticks(range(nT)); ax.set_xticklabels(SWP_TENORS, rotation=45, fontsize=7)
        ax.set_yticks(range(nE)); ax.set_yticklabels(SWP_EXPIRIES, fontsize=7)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

plot_loadings(res_static, f'Static PCA loadings — {BEST_COV}', n_show=5)

## 7 · Rolling PCA

Compares different window sizes and averaging strategies.

In [ ]:
# ── Fit configurations ───────────────────────────────────────────────────
rolling_configs = {
    '125d/step5/avg4': dict(window=125, step=5,  avg_windows=4),
    '252d/step5/avg4': dict(window=252, step=5,  avg_windows=4),
    '60d/step1/avg1':  dict(window=60,  step=1,  avg_windows=1),
}

rolling_models = {}
for name, kw in rolling_configs.items():
    m = RollingPCA(
        n_components    = 5,
        cov_method      = BEST_COV,
        ewma_lambda     = 0.96,
        auto_select_n   = True,
        target_variance = 0.95,
        **kw,
    )
    m.fit(X_swp, dates)
    rolling_models[name] = m
    res = m.get_loadings(TODAY)
    print(f'{name:22s}  →  {res.n_components} PCs  |  '
          f'{res.cumulative_variance_ratio[-1]*100:.2f}% var')

# ── Best rolling config (default: 125d/step5/avg4) ───────────────────────
BEST_ROLLING   = '125d/step5/avg4'
rolling_pca    = rolling_models[BEST_ROLLING]

In [ ]:
# ── How stable is variance explained over time? ───────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
fig.suptitle('Rolling PCA — cumulative variance explained over time', fontsize=12)

for name, m in rolling_models.items():
    df = m.all_explained_variance   # (windows × n_pcs)
    # Last column = cumulative for all PCs in that window
    ts = df.iloc[:, -1].astype(float)
    ax.plot(pd.to_datetime(ts.index), ts * 100, label=name, alpha=0.8)

ax.axhline(95, color=C_GREEN, ls=':', lw=1.2, label='95% target')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_xlabel('Window end date')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout(); plt.show()

## 8 · Iterated (regime-switching) PCA

Detects high-vol / low-vol regimes and fits a separate PCA per regime.
On a new day, loadings are a soft-weighted blend by posterior regime probability.

In [ ]:
iterated_pca = IteratedPCA(
    n_regimes       = 2,          # try 2 or 3
    n_components    = 5,
    cov_method      = BEST_COV,
    ewma_lambda     = 0.96,
    regime_features = 'vol_level', # 'vol_level' | 'pca_scores' | 'raw'
    blend           = True,        # soft blend (recommended)
    auto_select_n   = True,
    target_variance = 0.95,
)
iterated_pca.fit(X_swp, dates)

print('\nRegime summary:')
display(iterated_pca.regime_summary)

In [ ]:
# ── Regime timeline ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
fig.suptitle('Iterated PCA — regime detection', fontsize=12)

# Vol level
vol_series = pd.Series(
    np.abs(X_swp).mean(axis=1),
    index=dates[:len(X_swp)]
)
axes[0].fill_between(vol_series.index, vol_series, alpha=0.4, color=C_BLUE)
axes[0].set_ylabel('Mean |Δvol|')
axes[0].set_title('Vol activity')
axes[0].grid(True)

# Regime label
labels = iterated_pca.regime_labels_
colors_map = {0: C_BLUE, 1: C_ORANGE, 2: C_GREEN}
for r in range(iterated_pca.n_regimes):
    mask = labels == r
    axes[1].scatter(
        dates[:len(labels)][mask], [r]*mask.sum(),
        color=colors_map[r], s=4, label=f'Regime {r}'
    )
axes[1].set_ylabel('Regime'); axes[1].set_yticks(range(iterated_pca.n_regimes))
axes[1].legend(fontsize=9); axes[1].grid(True)

plt.tight_layout(); plt.show()

## 9 · P&L attribution — single model example

Run attribution for one date using the static PCA.

In [ ]:
attributor = PnLAttributor(prep)

# ── Static PCA attribution ───────────────────────────────────────────────
res_s  = static_pca.get_loadings()
report = attributor.attribute(
    pca_result     = res_s,
    vega_swp       = vega_swp,
    vol_change_swp = dv_swp,
    model_type     = 'Static',
)

print('── PC breakdown ─────────────────────────────────')
display(report.pc_table().round(3))
print('\n── Headline ─────────────────────────────────────')
display(report.headline())

# ── Waterfall chart ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
pcs  = list(report.pc_pnl.index)
vals = report.pc_pnl.values
cols = [C_GREEN if v >= 0 else C_RED for v in vals]
ax.bar(pcs, vals, color=cols, alpha=0.85, width=0.6)
ax.axhline(report.real_pnl, color=C_ORANGE, ls='--', lw=1.5, label=f'Real P&L = {report.real_pnl:.1f}')
ax.axhline(report.pca_pnl, color=C_BLUE,   ls=':',  lw=1.5, label=f'PCA P&L  = {report.pca_pnl:.1f}')
ax.set_title('P&L by PC — Static PCA'); ax.legend(); ax.grid(True, axis='y')
ax.set_ylabel('P&L')
plt.tight_layout(); plt.show()

## 10 · Model comparison — all three variants

Side-by-side P&L attribution error for Static, Rolling, and Iterated PCA.

In [ ]:
cmp = PCAModelComparison(
    preprocessor    = prep,
    n_components    = 5,
    target_variance = 0.95,
    cov_method      = BEST_COV,
    ewma_lambda     = 0.96,
    rolling_window  = 125,
    rolling_step    = 5,
    rolling_avg     = 4,
    n_regimes       = 2,
)
cmp.fit(X_swp, dates)

In [ ]:
comparison = cmp.compare(vega_swp, dv_swp, today=TODAY)
print('\n── Model comparison ─────────────────────────────────────────────────')
display(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'Model comparison — {TODAY}', fontsize=12)

models = comparison.index.tolist()
errors = comparison['Error'].astype(float).values
ep     = [float(e.strip('%')) for e in comparison['Error %'].values]

# Absolute error
ax = axes[0]
cols = [C_GREEN if abs(e) < abs(errors).max()*0.5 else C_ORANGE for e in errors]
ax.bar(models, errors, color=cols, alpha=0.85, width=0.4)
ax.axhline(0, color=C_GRAY, lw=0.8)
ax.set_title('Attribution error'); ax.set_ylabel('P&L error'); ax.grid(True, axis='y')

# Error %
ax = axes[1]
cols = [C_GREEN if e < 15 else (C_ORANGE if e < 30 else C_RED) for e in ep]
ax.bar(models, ep, color=cols, alpha=0.85, width=0.4)
ax.axhline(15, color=C_ORANGE, ls='--', lw=1, label='15% threshold')
ax.set_title('Error %'); ax.set_ylabel('%'); ax.legend(); ax.grid(True, axis='y')

plt.tight_layout(); plt.show()

## 11 · Cap floor — separate PCA (recommended)

Run a dedicated PCA on the cap floor surface and add its P&L separately,
rather than injecting it into the swaption 1Y pillar.

In [ ]:
if cfg.separate_pca:
    # ── Fit cap floor PCA ────────────────────────────────────────────────
    cap_static = StaticPCA(
        n_components    = 3,
        cov_method      = BEST_COV,
        ewma_lambda     = 0.96,
        auto_select_n   = True,
        target_variance = 0.95,
    )
    cap_static.fit(X_cap)
    res_cap = cap_static.result_
    print(f'Cap floor PCA: {res_cap.n_components} PCs → '
          f'{res_cap.cumulative_variance_ratio[-1]*100:.1f}% var')

    # ── Attribute P&L separately ─────────────────────────────────────────
    report_swp = attributor.attribute(res_static, vega_swp, dv_swp, 'Static (swaptions)')
    report_cap = attributor.attribute(res_cap,    vega_cap, dv_cap,  'Static (caps)')

    combined_pca  = report_swp.pca_pnl  + report_cap.pca_pnl
    combined_real = report_swp.real_pnl  + report_cap.real_pnl
    combined_err  = combined_real - combined_pca

    print(f'\nCombined (swaptions + caps):')
    print(f'  Real P&L : {combined_real:.3f}')
    print(f'  PCA P&L  : {combined_pca:.3f}')
    print(f'  Error    : {combined_err:.3f}  ({abs(combined_err/combined_real)*100:.1f}%)')
else:
    print('separate_pca=False — skipping cap floor attribution')

## 12 · Sensitivity analysis — EWMA lambda & window size

Sweep over key parameters to find the configuration with lowest attribution error.

In [ ]:
# ── EWMA lambda sweep (Static PCA) ───────────────────────────────────────
lambdas = [0.90, 0.92, 0.94, 0.96, 0.97, 0.98, 0.99]
errors_lambda = []

for lam in lambdas:
    m = StaticPCA(n_components=5, cov_method='ewma_lw',
                  ewma_lambda=lam, auto_select_n=True, target_variance=0.95)
    m.fit(X_swp)
    r = attributor.attribute(m.get_loadings(), vega_swp, dv_swp)
    errors_lambda.append(r.error_pct * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Sensitivity analysis', fontsize=12)

ax = axes[0]
ax.plot(lambdas, errors_lambda, color=C_BLUE, marker='o', ms=6)
best_lam = lambdas[np.argmin(errors_lambda)]
ax.axvline(best_lam, color=C_GREEN, ls='--', lw=1.2,
           label=f'Best λ={best_lam}')
ax.set_xlabel('EWMA lambda'); ax.set_ylabel('Attribution error %')
ax.set_title('EWMA lambda (Static PCA)'); ax.legend(); ax.grid(True)

# ── Window size sweep (Rolling PCA) ──────────────────────────────────────
windows = [60, 90, 125, 180, 252]
errors_window = []

for win in windows:
    m = RollingPCA(window=win, step=5, n_components=5,
                   cov_method=BEST_COV, avg_windows=4,
                   auto_select_n=True, target_variance=0.95)
    m.fit(X_swp, dates)
    res = m.get_loadings(TODAY)
    r = attributor.attribute(res, vega_swp, dv_swp)
    errors_window.append(r.error_pct * 100)

ax = axes[1]
ax.plot(windows, errors_window, color=C_ORANGE, marker='s', ms=6)
best_win = windows[np.argmin(errors_window)]
ax.axvline(best_win, color=C_GREEN, ls='--', lw=1.2,
           label=f'Best window={best_win}d')
ax.set_xlabel('Window size (days)'); ax.set_ylabel('Attribution error %')
ax.set_title('Rolling window size'); ax.legend(); ax.grid(True)

plt.tight_layout(); plt.show()
print(f'Best lambda: {best_lam}  |  Best window: {best_win}d')

## 13 · Next steps

Once this baseline is running with real data:

| Step | Action |
|------|--------|
| **Step 2** | Swap `SurfaceConfig(separate_pca=True)` and compare combined real P&L error |
| **Step 3** | Run sensitivity grid: `cov_method × ewma_lambda × window` and pick the winner |
| **Step 4** | Add second-order correction: `½ · volga · (Δvol)²` in `PnLAttributor.attribute()` |
| **Step 5** | Regress real P&L on PC factor scores to calibrate beta biases |
| **Step 6** | Promote best config to production with a weekly rolling re-fit job |